# Apps from Models - HiGHS

This notebook shows how to:

1. Create a decision model that solves a knapsack problem with the
   HiGHS solver.
2. Run it locally.
3. Push it to a Nextmv Cloud Application.
4. Run it remotely.

Let’s dive right in! 🐰

# Dependencies

Install the necessary Python packages.



In [ ]:
%pip install highspy
%pip install "nextmv[all]"

# Imports

Add the necessary imports.

In [2]:
import os
import json
import time
from importlib.metadata import version

import highspy
import nextmv
import nextmv.cloud

# 1. Create the decision model

Use HiGHS to solve a classic MIP with the `nextmv.Model` class.

In [3]:
class DecisionModel(nextmv.Model):
    def solve(self, input: nextmv.Input) -> nextmv.Output:
        """Solves the given problem and returns the solution."""

        start_time = time.time()

        # Creates the solver.
        solver = highspy.Highs()
        solver.silent()  # Solver output ignores stdout redirect, silence it.
        solver.setOptionValue("time_limit", input.options.duration)

        # Initializes the linear sums.
        weights = 0.0
        values = 0.0

        # Creates the decision variables and adds them to the linear sums.
        items = []
        for item in input.data["items"]:
            item_variable = solver.addVariable(0.0, 1.0, item["value"])
            items.append({"item": item, "variable": item_variable})
            weights += item_variable * item["weight"]
            values += item_variable * item["value"]

        # This constraint ensures the weight capacity of the knapsack will not be
        # exceeded.
        solver.addConstr(weights <= input.data["weight_capacity"])

        # Sets the objective function: maximize the value of the chosen items.
        status = solver.maximize(values)

        # Determines which items were chosen.
        chosen_items = [item["item"] for item in items if solver.val(item["variable"]) > 0.9]

        input.options.version = version("highspy")

        statistics = nextmv.Statistics(
            run=nextmv.RunStatistics(duration=time.time() - start_time),
            result=nextmv.ResultStatistics(
                value=sum(item["value"] for item in chosen_items),
                custom={
                    "status": str(status),
                    "variables": solver.numVariables,
                    "constraints": solver.numConstrs,
                },
            ),
        )

        return nextmv.Output(
            options=input.options,
            solution={"items": chosen_items},
            statistics=statistics,
        )

# 2. Run the model locally

Define the options that the model needs.

In [4]:
options = nextmv.Options(
    nextmv.Option("duration", int, 30, "Max runtime duration (in seconds).", False),
)

Instantiate the model.

In [5]:
model = DecisionModel()

Define some sample input data.

In [6]:
sample_input = {
  "items": [
    {
      "id": "cat",
      "value": 100,
      "weight": 20
    },
    {
      "id": "dog",
      "value": 20,
      "weight": 45
    },
    {
      "id": "water",
      "value": 40,
      "weight": 2
    },
    {
      "id": "phone",
      "value": 6,
      "weight": 1
    },
    {
      "id": "book",
      "value": 63,
      "weight": 10
    },
    {
      "id": "rx",
      "value": 81,
      "weight": 1
    },
    {
      "id": "tablet",
      "value": 28,
      "weight": 8
    },
    {
      "id": "coat",
      "value": 44,
      "weight": 9
    },
    {
      "id": "laptop",
      "value": 51,
      "weight": 13
    },
    {
      "id": "keys",
      "value": 92,
      "weight": 1
    },
    {
      "id": "nuts",
      "value": 18,
      "weight": 4
    }
  ],
  "weight_capacity": 50
}

Run the model locally.

In [ ]:
input = nextmv.Input(data=sample_input, options=options)
output = model.solve(input)
print(json.dumps(output.solution, indent=2))

# 3. Push the model to Nextmv Cloud

Convert the model to an application, hence the workflow name "Apps from
Models". Push the application to Nextmv Cloud.

Every app is production-ready with a full-featured API.

In [ ]:
client = nextmv.cloud.Client(api_key=os.getenv("NEXTMV_API_KEY"))
application = nextmv.cloud.Application(client=client, id="apps-from-models-highs-sample")

model_configuration = nextmv.ModelConfiguration(
    name="highs_model",
    requirements=[
        "highspy==1.9.0",
        "nextmv==1.6.2"
    ],
    options=options,
)
manifest = nextmv.cloud.Manifest.from_model_configuration(model_configuration)
application.push(
    manifest=manifest,
    model=model,
    model_configuration=model_configuration,
    verbose=True,
)

# 4. Run the model remotely

Execute an app run. This remote run produces an output that should be the same as the local run.

In [ ]:
result = application.new_run_with_result(input=sample_input, instance_id="devint")
print(json.dumps(result.output, indent=2))
